In [ ]:
!nvidia-smi

Tue Oct 27 05:56:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U trl peft datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.jsonl to train.jsonl


In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="train.jsonl", split="train")

# messages → prompt (system + user) and completion (assistant)
def to_prompt_completion(ex):
    return {"prompt": ex["messages"][:-1], "completion": ex["messages"][-1:]}

ds = ds.map(to_prompt_completion, remove_columns=["messages"])

split = ds.train_test_split(test_size=300, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(train_ds)
print(val_ds)
print(train_ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10479 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 10179
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 300
})
{'prompt': [{'role': 'system', 'content': "You are Shell Buddy. Convert the user's request into a single Linux bash command. Reply with only the command, no explanation and no markdown."}, {'role': 'user', 'content': 'Set up SSH connection forwarding in the background with no terminal or command execution from localhost port 8888 to "proxyhost" port 8888 and a reverse connection from "officefirewall" port 22222 to "localhost" port 22'}], 'completion': [{'role': 'assistant', 'content': 'ssh -fNT -L8888:proxyhost:8888 -R22222:localhost:22 officefirewall'}]}


In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",   # attach adapters to every linear layer
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir="shell-buddy-lora",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",    # learning rate slowly decreases toward the end
    warmup_steps=30,               # ...and starts small for the first 30 steps
    max_length=256,
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,  # small model, plenty of memory → faster without it
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,                # check validation loss every 100 steps
    save_strategy="no",            # we save manually at the end
    report_to="none",
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

... (8 outputs left out)

Building labels for eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,1.039156,0.974791,0.939638,0.758165,124467.000000
200,0.945866,0.911315,0.953054,0.768254,249267.000000
300,0.968875,0.874563,0.878316,0.777780,375170.000000
400,0.838092,0.833966,0.817124,0.789208,499788.000000
500,0.980302,0.814956,0.839635,0.793254,625149.000000
600,0.905696,0.785212,0.815642,0.798989,750745.000000
700,0.582723,0.792648,0.669473,0.795098,875267.000000
800,0.587034,0.773463,0.647824,0.799208,1000366.000000
900,0.623769,0.757238,0.646467,0.802613,1125180.000000
1000,0.635878,0.750841,0.635668,0.803179,1249892.000000


TrainOutput(global_step=1274, training_loss=0.7679284026895999, metrics={'train_runtime': 648.9362, 'train_samples_per_second': 31.371, 'train_steps_per_second': 1.963, 'total_flos': 4992181698700800.0, 'train_loss': 0.7679284026895999, 'epoch': 2.0})

In [ ]:
SYSTEM_PROMPT = (
    "You are Shell Buddy. Convert the user's request into a single Linux bash command. "
    "Reply with only the command, no explanation and no markdown."
)
model = trainer.model
tokenizer = trainer.processing_class
model.eval()

def ask(request):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": request}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for request in [
    "find all files bigger than 1GB in my home folder",
    "show which folders use the most disk space",
    "count lines in all python files in this directory",
    "delete empty directories under /tmp",
    "show the 10 most recently modified files",
]:
    print(f"{request}\n  → {ask(request)}\n")

find all files bigger than 1GB in my home folder
  → find ~ -size +1G

show which folders use the most disk space
  → df -h | awk '{print $1}' | sort -n | uniq -c | sort -nr

count lines in all python files in this directory
  → find . -name '*.py' | xargs wc -l

delete empty directories under /tmp
  → find /tmp -type d -empty -exec rmdir {} \;

show the 10 most recently modified files
  → find . -type f -printf "%T@ %p\n" | sort -n | head -n 10



In [ ]:
trainer.save_model("shell-buddy-lora-v2")
!zip -r shell-buddy-lora-v2.zip shell-buddy-lora-v2
files.download("shell-buddy-lora-v2.zip")


  adding: shell-buddy-lora-v2/ (stored 0%)
  adding: shell-buddy-lora-v2/README.md (deflated 65%)
  adding: shell-buddy-lora-v2/chat_template.jinja (deflated 71%)
  adding: shell-buddy-lora-v2/adapter_config.json (deflated 61%)
  adding: shell-buddy-lora-v2/training_args.bin (deflated 53%)
  adding: shell-buddy-lora-v2/adapter_model.safetensors (deflated 7%)
  adding: shell-buddy-lora-v2/tokenizer.json (deflated 81%)
  adding: shell-buddy-lora-v2/tokenizer_config.json (deflated 59%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>